# Outline

- Preparing Workspace

Importing packages, defining file paths, running user defined functions, setting API key, ...

- Preparing Imports

This section imports the "Census Configuration File.xlsx" and sets the user defined inputs to objects that are used for the API request

- Importing

This section requests data from the Census Bureau using the API

***

## Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_raw  = os.path.join(path_sp, 'Process Revamp', 'Task 9. Collect new data', 'Census')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Python Code', 'Census')
    path_config  = os.path.join(path_code, 'aa_config')
if user in ['jchoy', 'AAlAzzawi']:
    path_git     = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Python Code', 'Census')
    path_config  = os.path.join(path_code, 'aa_config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

***

## Preparing Imports

***

In [ ]:
print('Census Bureau processing parameters:')
print('')

# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying Census data
indicator_name     = df_params[df_params['Type'] == 'indicator_name' ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'       ]['Input'].values[0]
sample_type        = df_params[df_params['Type'] == 'sample'         ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'      ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'     ]['Input'].values[0]
margin_of_error    = df_params[df_params['Type'] == 'margin_of_error']['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'     ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'       ]['Input'].values[0]

# View
print('Indicator name:   ' + indicator_name )
print('Sample:           ' + sample_type    )
print('Estimate:         ' + estimate       )
print('Final geography:  ' + geography      )
print('Import geography: ' + import_tab     )
print('Margin of error:  ' + margin_of_error)
print('Start year:       ' + str(year_start))
print('End year:         ' + str(year_end  ))

In [ ]:
# Execute script to prepare API request inputs
exec(open(os.path.join(path_code, 'Step 01 - Queries', 'Step 01a - Prepare API Request Inputs.py')).read())

# View result
df_vars.head(3)

***

## Importing

***

In [ ]:
# Execute script to import Census Bureau Data
exec(open(os.path.join(path_code, 'Step 01 - Queries', 'Step 01b - Run API Queries.py')).read())

In [ ]:
# View result
print(df_census_raw.shape)
print(df_census_raw.Year.unique())
pd.set_option('display.max_columns', None)
display(df_census_raw)

In [ ]:
# Export Title
if geography == 'PUMA':
    export_title = '_'.join([indicator_name, geography, re.sub('ACS', 'PUMS', estimate)]) + '.csv'
else:
    export_title = '_'.join([indicator_name, geography, estimate]) + '.csv'


print("Exporting " + export_title + " to the following location: ")
print(path_raw)

# Export
df_census_raw.to_csv(os.path.join(path_raw, export_title), index = False)


print('')
print('Successfully exported!')